<a href="https://colab.research.google.com/github/laurakeita/MMM/blob/main/notebooks/05_synthetic_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 · Synthetic Data Validation
Generate a synthetic MMM dataset with known ground-truth ROI parameters, fit the model, and prepare outputs for attribution validation in notebook 06.

## Generate Synthetic Dataset

In [ ]:
from src.synthetic_data import generate_synthetic_mmm_data

df = generate_synthetic_mmm_data(n_weeks=104, seed=42)
df.to_csv('data/synthetic_mmm_data.csv', index=False)
print(df.shape)
df.head()

## Data Quality Check

In [ ]:
from src.validation import check_data_quality
from src.data_loader import detect_columns

media, impressions, output = detect_columns(df)
check_data_quality(df)
print('\nMedia:', media)
print('Impressions:', impressions)
print('KPI:', output)

## Fit Model on Synthetic Data

In [ ]:
import numpy as np, tensorflow_probability as tfp
from meridian import constants
from meridian.model import model, spec, prior_distribution
from src.data_loader import build_channel_mappings, build_meridian_dataset

cost_mapping, impressions_mapping = build_channel_mappings(media, impressions)
data_meridian = build_meridian_dataset(df, media, impressions, output, cost_mapping, impressions_mapping)

def estimate_lognormal_dist(mean, std):
    mu_log = np.log(mean) - 0.5 * np.log((std/mean)**2 + 1)
    std_log = np.sqrt(np.log((std/mean)**2 + 1))
    return mu_log, std_log

roi_mu_log, roi_sigma_log = estimate_lognormal_dist(2.0, 1.5)
prior = prior_distribution.PriorDistribution(
    roi_m=tfp.distributions.LogNormal(
        loc=np.float32(roi_mu_log), scale=np.float32(roi_sigma_log), name=constants.ROI_M)
)
model_spec = spec.ModelSpec(prior=prior, max_lag=7)
mmm = model.Meridian(input_data=data_meridian, model_spec=model_spec)
mmm.sample_prior(50)
mmm.sample_posterior(n_chains=3, n_adapt=500, n_burnin=500, n_keep=1000)

## Next Step
Run **`06_attribution_validation.ipynb`** to compare estimated ROI against ground-truth values.